# Getting over the dataset

The dataset keeps what is known of every post as JSON files, by platform and profile:
an index, each post, and what `delete` removed.

In [1]:
from amoorion import Amoorion

# The sandbox has its own dataset, in a temporary folder.
amo = Amoorion().use("dryrun")

# `[0]` is the result of the only platform; each carries its message.
first = amo.publish(text="first")[0]["message"]

second = amo.publish(text="second")[0]["message"]

# A snapshot copies the profile's posts into files.
amo.snapshot()

{'platform': 'dryrun',
 'profile': 'default',
 'path': '/tmp/amoorion-zx3k0ep0/dataset/dryrun/default',
 'read': 2,
 'complete': True,
 'present': 2,
 'gone': 0,
 'deleted': 0}

In [2]:
# One JSON file per post, and an index next to them.
for path in sorted(amo.dataset.path.rglob("*.json")):
    print(path.relative_to(amo.dataset.path))

dryrun/default/index.json
dryrun/default/posts/1.json
dryrun/default/posts/2.json


`delete` keeps a copy first, and the index remembers:

In [3]:
# Deleting on the platform keeps a copy first.
amo.delete(id=first["id"])["ok"]

# The index remembers: the deleted one is marked, not forgotten.
for entry in amo.dataset.index("dryrun", "default").values():
    print(
        entry.id,
        entry.state,
        entry.text,
    )

2 present second
1 deleted first


In [4]:
# The post still can be found, and its file is there.
(
    amo.dataset.find("dryrun", "default", first["id"]) is not None,
    sorted(
        p.name
        for p in amo.dataset.profile_dir("dryrun", "default").glob("deleted/*")
    ),
)

(True, ['1.json'])

It is plain files, so anything reads it, a search index of your own included:

In [5]:
# The dataset is plain files: any tool can read them, without amoorion.
import json

# The same index, by hand.
index = json.loads(
    (amo.dataset.profile_dir("dryrun", "default") / "index.json").read_text()
)

[(post["id"], post["state"]) for post in index["posts"]]

[('2', 'present'), ('1', 'deleted')]

## Where it can go

The dataset grows into the library of everything: posts and drafts, links, the graph of related personas, the storylines LLMs grow on. Being plain files, it can already be carried by anything. Two of the targets work today, without any new code; torrent, IPFS, FTP, S3, Hugging Face, Google Drive and SSHFS are planned for phase 8, with a vector database for moving through it quickly.

As a zip archive:


In [6]:
import shutil

import tempfile

import zipfile

from pathlib import Path

carrier = Path(tempfile.mkdtemp())

archive = shutil.make_archive(
    carrier / "dataset",
    "zip",
    amo.dataset.path,
)

zipfile.ZipFile(archive).namelist()[:4]

['dryrun/',
 'dryrun/default/',
 'dryrun/default/deleted/',
 'dryrun/default/posts/']

As a git repository, so every snapshot is a commit:


In [7]:
from git import Actor, Repo

# A repository of its own, next to the archive.
copy = shutil.copytree(
    amo.dataset.path,
    carrier / "repository",
)

repository = Repo.init(copy)

repository.git.add(all=True)

who = Actor(
    "amoorion",
    "amoorion@example.org",
)

commit = repository.index.commit(
    "snapshot",
    author=who,
    committer=who,
)

commit.message, len(repository.git.ls_files().splitlines())

('snapshot', 4)